# ML/AI Study Assistant — A Retrieval-Augmented Generation (RAG) Chatbot

A chatbot that answers questions about core Machine Learning / AI concepts by
retrieving relevant excerpts from a local knowledge base and grounding its
answers in them — a minimal, fully working implementation of the RAG pattern
used in real GenAI applications.

**Author:** Daniel Osei

## What this notebook demonstrates

- **Retrieval-augmented generation (RAG)**: answers are grounded in retrieved
  source text rather than the model's unguided output, reducing hallucination.
- **A local vector search pipeline**: text chunking, embedding, and similarity
  search using [Chroma](https://www.trychroma.com/) as the vector store.
- **An LLM call** via [Groq](https://groq.com/)'s free-tier API, with the
  response constrained to the retrieved context.
- **Source attribution**: every answer shows which knowledge-base articles it
  drew on.

## How it works

```
User question
     │
     ▼
Embed question (TF-IDF vectorizer, fit on the corpus)
     │
     ▼
Similarity search against Chroma vector store  →  top-k relevant chunks
     │
     ▼
Build a prompt: "Using only this context, answer the question"
     │
     ▼
Groq LLM (Llama 3.1) generates the answer
     │
     ▼
Answer + cited sources
```

## Setup before running this notebook

1. `pip install -r requirements.txt`
2. Get a **free** Groq API key at [console.groq.com/keys](https://console.groq.com/keys)
   (no credit card required).
3. Copy `.env.example` to `.env` and paste your key in. `.env` is gitignored —
   your key never gets committed.

Run the cells top to bottom. Sections 1–3 (loading, chunking, indexing,
retrieval) work with no API key at all. Section 4 (generating answers) needs
your Groq key.


## 1. Setup

In [ ]:
import os
import re
import pickle
from pathlib import Path

import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a local .env file, if present

DATA_DIR = Path("data")
CHROMA_DIR = Path("chroma_db")
VECTORIZER_PATH = CHROMA_DIR / "tfidf_vectorizer.pkl"
COLLECTION_NAME = "ml_ai_knowledge_base"

CHROMA_DIR.mkdir(exist_ok=True)
print("Data directory:", DATA_DIR.resolve())
print("Knowledge base files found:", len(list(DATA_DIR.glob("*.txt"))))


## 2. Load and chunk the knowledge base

The `data/` folder contains study notes on 12 core ML/AI topics, adapted from
Wikipedia (CC BY-SA 4.0 — see the source line at the top of each file). If you
want to refresh this content or add your own topics, see the optional
**"Refreshing the knowledge base from Wikipedia"** section near the end of
this notebook.

Each article gets split into overlapping ~800-character chunks so that
retrieval can return focused, relevant excerpts rather than whole articles.


In [ ]:
CHUNK_SIZE = 800       # characters
CHUNK_OVERLAP = 150    # characters of overlap between consecutive chunks
MIN_CHUNK_CHARS = 200  # trailing fragments shorter than this get merged into
                        # the previous chunk instead of indexed alone -- a short
                        # fragment like "...standing itself." carries almost no
                        # TF-IDF signal and can even embed as an all-zero vector,
                        # which distorts distance-based ranking (see note below).


def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def chunk_text(text: str, source: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            if chunks and len(chunk) < MIN_CHUNK_CHARS:
                chunks[-1]["text"] = (chunks[-1]["text"] + " " + chunk).strip()
            else:
                chunks.append({"text": chunk, "source": source})
        start += chunk_size - overlap
    return chunks


def load_corpus():
    txt_files = sorted(DATA_DIR.glob("*.txt"))
    all_chunks = []
    for path in txt_files:
        raw = path.read_text(encoding="utf-8")
        cleaned = clean_text(raw)
        first_line = cleaned.splitlines()[0] if cleaned else path.stem
        source_label = first_line.replace("Source: ", "") if first_line.startswith("Source:") else path.stem
        chunks = chunk_text(cleaned, source=source_label)
        all_chunks.extend(chunks)
        print(f"  {path.name}: {len(cleaned):,} chars -> {len(chunks)} chunks")
    return all_chunks


all_chunks = load_corpus()
print(f"\nTotal chunks: {len(all_chunks)}")


## 3. Build the vector index (TF-IDF + Chroma)

**Why TF-IDF instead of a neural embedding model (e.g. sentence-transformers)?**
TF-IDF is a classic, fully local sparse-embedding technique from scikit-learn —
no multi-hundred-MB model download, no dependency on Hugging Face being
reachable, works the moment you `pip install`. Retrieval quality is a notch
below dense neural embeddings on more diverse corpora, but for a focused,
single-domain knowledge base like this one it performs well (verified against
test questions in Section 3.2 below).

**Upgrade path:** for a more diverse corpus, swap the `TfidfVectorizer` step
below for a `sentence-transformers` model via Chroma's
`SentenceTransformerEmbeddingFunction` — the rest of the pipeline (chunking,
storage, retrieval, prompting) stays identical. That's the benefit of keeping
retrieval and generation cleanly separated in a RAG system.


In [ ]:
documents = [c["text"] for c in all_chunks]
metadatas = [{"source": c["source"]} for c in all_chunks]

print("Fitting TF-IDF vectorizer on the corpus (fully local, no downloads)...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
embeddings_arr = vectorizer.fit_transform(documents).toarray()

# Safety net: drop any degenerate all-zero-vector chunk (see MIN_CHUNK_CHARS
# note above) -- a zero vector can look artificially "close" to every query
# under L2 distance and pollute retrieval results.
norms = (embeddings_arr ** 2).sum(axis=1) ** 0.5
keep = norms > 0
if (~keep).sum():
    print(f"  Dropping {(~keep).sum()} degenerate zero-vector chunk(s)")
documents = [d for d, k in zip(documents, keep) if k]
metadatas = [m for m, k in zip(metadatas, keep) if k]
embeddings = embeddings_arr[keep].tolist()
ids = [f"chunk_{i}" for i in range(len(documents))]

with open(VECTORIZER_PATH, "wb") as f:
    pickle.dump(vectorizer, f)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(name=COLLECTION_NAME)

batch_size = 100
for i in range(0, len(documents), batch_size):
    collection.add(
        ids=ids[i:i + batch_size],
        documents=documents[i:i + batch_size],
        metadatas=metadatas[i:i + batch_size],
        embeddings=embeddings[i:i + batch_size],
    )

print(f"Indexed {collection.count()} chunks into Chroma at {CHROMA_DIR}/")


### 3.2 Sanity-check retrieval before wiring up the LLM

Always verify the retrieval step on its own -- if this returns the wrong chunks, no amount of prompt engineering downstream will fix the final answers.

In [ ]:
def retrieve(question: str, n_results: int = 4):
    query_embedding = vectorizer.transform([question]).toarray().tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n_results)
    chunks = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        chunks.append({"text": doc, "source": meta["source"], "distance": dist})
    return chunks


test_queries = [
    "What is the difference between supervised and unsupervised learning?",
    "How does backpropagation work in a neural network?",
    "Why do transformers use self-attention instead of recurrence?",
    "What causes overfitting and how can I prevent it?",
    "How does RAG reduce hallucination in LLMs?",
]

for q in test_queries:
    top = retrieve(q, n_results=1)[0]
    print(f"Q: {q}")
    print(f"  -> top match: [{top['source'][:60]}] (distance={top['distance']:.3f})")
    print(f"     {top['text'][:100]!r}...\n")


Each test question's top match comes from the correct source article, which
confirms the embedding + retrieval pipeline is working correctly before we
add the LLM on top of it.


## 4. Generate grounded answers with Groq

This is the "generation" half of RAG: the retrieved chunks get inserted into
a prompt that instructs the LLM to answer *using only that context*. This
needs a free Groq API key (see Setup at the top of this notebook) — if
`GROQ_API_KEY` isn't set, the cell below will explain what to do instead of
failing silently.


In [ ]:
from groq import Groq

GROQ_MODEL = "llama-3.1-8b-instant"  # fast + free-tier friendly; try
                                       # "llama-3.3-70b-versatile" for higher quality

SYSTEM_PROMPT = """You are a helpful study-assistant chatbot that answers questions about \
machine learning and AI concepts using ONLY the provided context excerpts. \
Rules:
- Base your answer strictly on the context below. Do not use outside knowledge.
- If the context does not contain enough information to answer, say so clearly \
instead of guessing.
- Keep answers clear and well-organized, as if explaining to a Masters student.
- When helpful, mention which topic(s) the information came from.
"""


def build_prompt(question: str, chunks: list) -> str:
    context = "\n\n".join(
        f"[Excerpt {i+1} - source: {c['source']}]\n{c['text']}" for i, c in enumerate(chunks)
    )
    return f"Context excerpts:\n\n{context}\n\nQuestion: {question}\n\nAnswer the question using only the context excerpts above."


def answer_question(question: str, n_results: int = 4):
    chunks = retrieve(question, n_results=n_results)
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        print("No GROQ_API_KEY found -- set it in your .env file (see .env.example) to enable answer generation.")
        print("Get a free key at: https://console.groq.com/keys")
        return None, chunks

    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(question, chunks)},
        ],
        temperature=0.2,
        max_tokens=600,
    )
    return completion.choices[0].message.content, chunks


### 4.1 Try it out

Edit `my_question` below and re-run the cell to ask anything covered by the knowledge base (see the topic list in `data/`).

In [ ]:
my_question = "Why do transformers use self-attention instead of recurrence?"

answer, used_chunks = answer_question(my_question)

if answer:
    print("Q:", my_question)
    print("\nA:", answer)
    print("\nSources used:")
    for s in sorted({c['source'] for c in used_chunks}):
        print(" -", s)


## 5. (Optional) Refreshing the knowledge base from Wikipedia

The `data/` folder already ships with 12 topics pre-populated, so you don't
need to run this section to use the notebook. Run it only if you want to
pull fresh article text or change which topics are included — it needs
internet access to `en.wikipedia.org` (no API key required).

After running this cell, re-run Sections 2 and 3 above to rebuild the index
with the refreshed content.


In [ ]:
import time
import requests

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"

# Edit this list to change which topics make up the knowledge base.
WIKIPEDIA_TOPICS = [
    "Machine learning", "Supervised learning", "Unsupervised learning",
    "Reinforcement learning", "Artificial neural network", "Deep learning",
    "Convolutional neural network", "Transformer (deep learning architecture)",
    "Large language model", "Natural language processing", "Overfitting",
    "Retrieval-augmented generation",
]


def fetch_article_text(title: str) -> str:
    params = {"action": "query", "prop": "extracts", "explaintext": 1, "format": "json", "titles": title, "redirects": 1}
    headers = {"User-Agent": "ml-rag-chatbot-demo/1.0 (educational project)"}
    resp = requests.get(WIKIPEDIA_API_URL, params=params, headers=headers, timeout=20)
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    return next(iter(pages.values())).get("extract", "")


def refresh_wikipedia_data():
    for topic in WIKIPEDIA_TOPICS:
        try:
            raw = fetch_article_text(topic)
            if not raw:
                print(f"  ! No content for '{topic}'")
                continue
            cleaned = re.split(r"\n==\s*(See also|References|External links|Further reading|Notes)\s*==", raw)[0]
            cleaned = re.sub(r"\n{2,}", "\n\n", cleaned).strip()
            header = f"Source: Wikipedia - {topic} (https://en.wikipedia.org/wiki/{topic.replace(' ', '_')}), adapted under CC BY-SA 4.0\n\n"
            filename = topic.lower().replace(' ', '_').replace('(', '').replace(')', '') + '.txt'
            (DATA_DIR / filename).write_text(header + cleaned, encoding='utf-8')
            print(f"  wrote {filename} ({len(cleaned):,} chars)")
            time.sleep(0.2)
        except Exception as e:
            print(f"  ! Failed to fetch '{topic}': {e}")


# Uncomment to actually refresh the data:
# refresh_wikipedia_data()
print("Uncomment the call above to refresh data/ from live Wikipedia.")


## Attribution & limitations

The knowledge base in `data/` is adapted from Wikipedia articles, available
under the [Creative Commons Attribution-ShareAlike 4.0 License](https://creativecommons.org/licenses/by-sa/4.0/).
Each file cites its source article and URL in its header line.

**Known limitations:**
- The knowledge base covers 12 core ML/AI topics — questions outside that
  scope correctly get a "not enough information" response rather than a
  fabricated answer, since the LLM is instructed to only use retrieved context.
- TF-IDF retrieval is keyword/n-gram based rather than semantic, so heavily
  paraphrased questions that share little vocabulary with the source text may
  retrieve weaker matches than a dense-embedding model would (see the upgrade
  note in Section 3).


## 6. (Optional) Interactive Q&A loop

Run this cell to ask multiple questions in a row without re-running the
notebook. Enter a blank line to stop. Placed last so it doesn't block
"Run All" from reaching the rest of the notebook if you don't want to type
anything right now.


In [ ]:
while True:
    q = input("Ask a question (blank to stop): ").strip()
    if not q:
        break
    answer, used_chunks = answer_question(q)
    if answer:
        print("\n" + answer + "\n")
        print("Sources:", ", ".join(sorted({c['source'] for c in used_chunks})))
    print("-" * 60)
